# LizyML Widget Tutorial — Multiclass Classification

This tutorial demonstrates using LizyML Widget for a **multiclass classification** task
with the [Wine](https://scikit-learn.org/stable/datasets/toy_dataset.html#wine-recognition-dataset) dataset.

You will learn:
1. Loading real-world data into the widget
2. Reviewing auto-detected settings for multiclass tasks
3. Configuring evaluation metrics appropriate for multiclass
4. Reviewing multiclass-specific metrics and plots
5. Hyperparameter tuning
6. Running inference with multi-class predictions

## 0. Installation

```bash
pip install lizyml-widget lizyml[plots,tuning,calibration,explain]
```

## 1. Load the Wine Dataset

This dataset contains 178 samples with 13 chemical features from
three different wine cultivars grown in Italy.
The target has 3 classes (cultivar 0, 1, 2).

In [ ]:
import pandas as pd
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
df = data.frame
print(f"Shape: {df.shape}")
print(f"Classes: {sorted(df['target'].unique())}")
print(f"Target distribution:\n{df['target'].value_counts().sort_index()}")
df.head()

## 2. Launch the Widget

In [ ]:
from lizyml_widget import LizyWidget

w = LizyWidget()
w.load(df, target="target")
w

## 3. Data Tab — Auto-Detection

The widget auto-detects:
- **Task**: `multiclass` (3 unique target values, integer type)
- **CV**: `stratified_kfold` (preserves class balance across folds)
- **Columns**: All 13 chemical features included

In [ ]:
print(f"Task:     {w.task}")
print(f"CV:       {w.cv_method} ({w.cv_n_splits} folds)")
print(f"Shape:    {w.df_shape}")

## 4. Config — Multiclass Settings

For multiclass, we configure:
- `objective: multiclass` (set automatically by task detection)
- Appropriate evaluation metrics (AUC, F1, accuracy)

In [ ]:
w.set_config({
    "model": {
        "name": "lgbm",
        "params": {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "max_depth": 4,
        },
    },
    "training": {
        "seed": 42,
        "early_stopping": {"enabled": True, "rounds": 30},
    },
    "evaluation": {
        "metrics": ["auc", "f1", "accuracy", "logloss"],
    },
})

print("Config set with AUC, F1, Accuracy, LogLoss for multiclass")

## 5. Run Fit

In [ ]:
w.fit()

## 6. Results — Multiclass Metrics & Plots

After fitting, the Results tab shows:

**Metrics**:
- **AUC** — One-vs-Rest area under ROC (macro-averaged)
- **F1** — F1 Score (macro-averaged)
- **Accuracy** — Overall classification accuracy
- **LogLoss** — Multi-class logarithmic loss

**Multiclass plots**:
- **ROC Curve** — One-vs-Rest ROC for each class
- **OOF Distribution** — Out-of-fold prediction distribution per class
- **Feature Importance** — Split, Gain, and SHAP variants

In [ ]:
summary = w.get_fit_summary()
if summary:
    print(f"Fold count: {summary.fold_count}")
    for name, vals in summary.metrics.items():
        if isinstance(vals, dict):
            oos = vals.get("oos", "N/A")
            print(f"  {name}: OOS={oos}")

## 7. Tune — Hyperparameter Optimization

In [ ]:
w.tune()

tune_summary = w.get_tune_summary()
if tune_summary:
    print(f"Best score:  {tune_summary.best_score:.4f}")
    print(f"Metric:      {tune_summary.metric_name} ({tune_summary.direction})")
    print(f"Best params: {tune_summary.best_params}")
    print(f"Trials:      {len(tune_summary.trials)}")

## 8. Inference — Multiclass Predictions

For multiclass, predictions include class probabilities for each class.

In [ ]:
df_test = df.drop(columns=["target"]).tail(10)

result = w.predict(df_test)
if result:
    print(f"Predictions shape: {result.predictions.shape}")
    print(f"Columns: {list(result.predictions.columns)}")
    result.predictions.head()

## 9. Direct Model Access

You can access the underlying LizyML Model object for advanced operations.

In [ ]:
model = w.get_model()
if model:
    print(f"Model type: {type(model).__name__}")
    # Direct LizyML operations:
    # model.plot_learning_curve().show()
    # model.importance_plot(kind="shap").show()

## 10. Save Config

In [ ]:
w.save_config("multiclass_config.yaml")
print("Config saved to multiclass_config.yaml")

## 11. Export Code

In [ ]:
# Export standalone training code (no LizyML dependency needed)
export_path = w.export_code("./exported_code")
print(f"Code exported to: {export_path}")
print("Generated files:")
for f in sorted(export_path.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(export_path)}")